# Fine-tune SuryaOCR recognition model on the DACK Vietnamese dataset

Run from the `SinoNom-NLP` repository. Uses the same dataset as
`vietocr_dataset_cleanup_rtx5090.ipynb` (`./data/DACK/data`,
`rec_train.txt` / `rec_val.txt` / `rec_test.txt`, `vi_dict.txt`, `train/val/test` image folders).

**Important - read before running:** the current published `surya-ocr` package (v2, the
`pip install surya-ocr` you get today) was rearchitected into a single ~650M-param VLM
served through `vllm`/`llama.cpp`, and its maintainers state there is no public
fine-tuning pipeline for it (they ask you to contact them directly for custom training).
This notebook instead pins the older `surya-ocr==0.8.0` release, which still ships the
original trainable recognition model (a modified Donut/Swin encoder + ByT5-style
UTF-16 decoder). There is no official training script even in that version - the
training loop here is reverse-engineered from `surya/recognition.py` and
`surya/model/recognition/*` in that release to replicate the exact
encoder -> text-encoder -> decoder path used at inference, so the fine-tuned weights
stay loadable by `surya.model.recognition.model.load_model()`.

This is a best-effort, unofficial community fine-tune - not a supported Datalab workflow.

## 1. Environment and GPU check

In [1]:
import os
import sys
import subprocess

print("Python executable:", sys.executable)
subprocess.run(["nvidia-smi"], check=False)

# Pin the last surya-ocr release before the v2 VLM rewrite - this is the version whose
# recognition model is a plain trainable PyTorch / transformers model.
# transformers must also be pinned close to what surya-ocr==0.8.0 was built against
# (pyproject pins "^4.41.0"); newer transformers changed PretrainedConfig.to_diff_dict()
# to instantiate `self.__class__()` with no args when repr'ing/logging a config, which
# raises `KeyError: 'encoder'` in SuryaOCRConfig.__init__ since encoder/decoder are
# mandatory kwargs there.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "surya-ocr==0.8.0",
        "transformers==4.41.2",
        "pandas", "rapidfuzz", "tqdm", "matplotlib",
    ],
    check=True,
)
print("Dependencies installed")


Python executable: /workspace/.venv/bin/python
Mon Aug 31 06:32:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        Off |   00000000:01:00.0 Off |                  N/A |
|  0%   42C    P8             26W /  600W |    7756MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+

Dependencies installed



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    print("Device:", torch.cuda.get_device_name(idx))
    print("Capability:", torch.cuda.get_device_capability(idx))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# CUDA smoke test
a = torch.randn(1024, 1024, device=DEVICE, dtype=DTYPE)
b = torch.randn(1024, 1024, device=DEVICE, dtype=DTYPE)
c = a @ b
torch.cuda.synchronize() if torch.cuda.is_available() else None
print("Smoke test matmul result shape:", c.shape, "device:", c.device)

torch: 2.9.0+cu128
CUDA available: True
Device: NVIDIA GeForce RTX 5090
Capability: (12, 0)
Smoke test matmul result shape: torch.Size([1024, 1024]) device: cuda:0


## 2. Config

In [3]:
from pathlib import Path

REPO_ROOT = Path.cwd()
DACK_DATA_DIR = REPO_ROOT / "data" / "DACK" / "data"
assert DACK_DATA_DIR.exists(), f"Dataset dir not found: {DACK_DATA_DIR}"

TRAIN_LABELS = DACK_DATA_DIR / "rec_train.txt"
VAL_LABELS = DACK_DATA_DIR / "rec_val.txt"
TEST_LABELS = DACK_DATA_DIR / "rec_test.txt"
DICT_FILE = DACK_DATA_DIR / "vi_dict.txt"

OUTPUT_DIR = REPO_ROOT / "output" / "suryaocr_finetune"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

LANG = "vi"
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
FREEZE_ENCODER = True  # freeze the Donut/Swin vision encoder, only fine-tune text_encoder + decoder
MAX_TRAIN_SAMPLES = None  # set an int to subsample for a quick smoke run
LOG_EVERY = 50
VALIDATE_EVERY_EPOCH = True
VAL_SUBSET_SIZE = 200  # number of val examples used for CER during training (full autoregressive decoding is slow)
TEST_SUBSET_SIZE = 200  # number of test examples used to benchmark the base SuryaOCR model

print("DACK_DATA_DIR:", DACK_DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DACK_DATA_DIR: /workspace/SinoNom-NLP/data/DACK/data
OUTPUT_DIR: /workspace/SinoNom-NLP/output/suryaocr_finetune


## 3. Load dataset labels

In [4]:
def load_labels(label_file: Path, data_dir: Path):
    examples = []
    with open(label_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            rel_path, text = line.split("\t", 1)
            img_path = data_dir / rel_path
            examples.append((img_path, text))
    return examples

train_examples = load_labels(TRAIN_LABELS, DACK_DATA_DIR)
val_examples = load_labels(VAL_LABELS, DACK_DATA_DIR)
test_examples = load_labels(TEST_LABELS, DACK_DATA_DIR)

if MAX_TRAIN_SAMPLES:
    train_examples = train_examples[:MAX_TRAIN_SAMPLES]

print(f"train: {len(train_examples)} | val: {len(val_examples)} | test: {len(test_examples)}")
print("Sample:", train_examples[0])

train: 93997 | val: 3000 | test: 3003
Sample: (PosixPath('/workspace/SinoNom-NLP/data/DACK/data/train/letrieulichkhoatiensitap2__page_037_crop_14.jpg'), '« Cô dĩ thử khoa nhi lịch sử chi : Có hữu tại phụ tử đồng')


## 4. Load Surya recognition model and processor

In [5]:
from surya.model.recognition.model import load_model as load_rec_model
from surya.model.recognition.processor import load_processor as load_rec_processor

model = load_rec_model()
processor = load_rec_processor()
model = model.to(DEVICE, dtype=DTYPE)
model.train()

tokenizer = processor.tokenizer
pad_id = tokenizer.pad_id
eos_id = tokenizer.eos_id  # also used as BOS
decoder_start_token_id = model.config.decoder_start_token_id
query_token_count = model.text_encoder.config.query_token_count
vocab_size = model.decoder.config.vocab_size

print("pad_id:", pad_id, "eos_id:", eos_id, "decoder_start_token_id:", decoder_start_token_id)
print("query_token_count:", query_token_count, "vocab_size:", vocab_size)

if FREEZE_ENCODER:
    for p in model.encoder.parameters():
        p.requires_grad = False
    print("Encoder frozen; fine-tuning text_encoder + decoder only")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,}")

/workspace/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded recognition model vikp/surya_rec2 on device cuda with dtype torch.float16
pad_id: 0 eos_id: 1 decoder_start_token_id: 1
query_token_count: 128 vocab_size: 65792
Encoder frozen; fine-tuning text_encoder + decoder only
Trainable params: 380,971,520 / 469,831,032


## 5. Dataset / DataLoader

`Byt5LangTokenizer` does not pad automatically, so batching is handled manually in the
collate function below (pixel values are stacked, label token sequences are right-padded
with `pad_id`).

In [6]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class SuryaOCRDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        img_path, text = self.examples[idx]
        image = Image.open(img_path).convert("RGB")
        return image, text


def make_collate_fn(processor, lang, pad_id):
    image_processor = processor.image_processor
    tokenizer = processor.tokenizer

    def collate_fn(batch):
        images, texts = zip(*batch)

        pixel_values = image_processor(list(images), return_tensors="pt")["pixel_values"]

        encodings = tokenizer(texts=list(texts), langs=[[lang]] * len(texts))
        token_lists = encodings["input_ids"]

        max_len = max(len(t) for t in token_lists)
        labels = torch.full((len(token_lists), max_len), pad_id, dtype=torch.long)
        for i, toks in enumerate(token_lists):
            labels[i, : len(toks)] = torch.tensor(toks, dtype=torch.long)

        return pixel_values, labels

    return collate_fn


collate_fn = make_collate_fn(processor, LANG, pad_id)

train_dataset = SuryaOCRDataset(train_examples)
val_dataset = SuryaOCRDataset(val_examples)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_fn, num_workers=4, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_fn, num_workers=2,
)

print("train batches:", len(train_loader), "| val batches:", len(val_loader))

train batches: 11749 | val batches: 375


## 5.5 Baseline test-set evaluation (base SuryaOCR)

Measure the unmodified pretrained SuryaOCR model on the DACK test split before fine-tuning.

In [10]:
from rapidfuzz.distance import Levenshtein
from surya.recognition import batch_recognition

model.eval()

test_subset = test_examples[:TEST_SUBSET_SIZE] if TEST_SUBSET_SIZE else test_examples
test_images = [Image.open(p).convert("RGB") for p, _ in test_subset]
test_texts = [t for _, t in test_subset]
test_langs = [[LANG]] * len(test_subset)

with torch.no_grad():
    test_predictions, _ = batch_recognition(test_images, test_langs, model, processor)

total_chars = 0
total_dist = 0
total_correct = 0
for pred_text, gt in zip(test_predictions, test_texts):
    edit_dist = Levenshtein.distance(pred_text, gt)
    total_dist += edit_dist
    total_chars += max(len(gt), 1)
    total_correct += int(pred_text == gt)

base_cer = total_dist / total_chars
base_acc = total_correct / len(test_predictions)
base_norm_edit = total_dist / sum(max(len(gt), 1) for gt in test_texts)
print(f"Base SuryaOCR test CER on {len(test_subset)} samples: {base_cer:.4f}")

print(f"Base SuryaOCR test Accuracy on {len(test_subset)} samples: {base_acc:.4f}")    
print("---")

print(f"Base SuryaOCR test Normalized Edit Distance on {len(test_subset)} samples: {base_norm_edit:.4f}")    
print("PR :", pred_text)

for pred_text, gt in list(zip(test_predictions, test_texts))[:5]:
    print("PR :", pred_text)
    print("GT :", gt)

Recognizing Text:   0%|          | 0/1 [00:00<?, ?it/s]

Recognizing Text: 100%|██████████| 1/1 [00:01<00:00,  1.53s/it]

Base SuryaOCR test CER on 200 samples: 0.0342
Base SuryaOCR test Accuracy on 200 samples: 0.4300
---
Base SuryaOCR test Normalized Edit Distance on 200 samples: 0.0342
PR : tự nhiên bắt được hắn; khanh về, chỉ nên sai người mật báo thôi!
PR : Cân-sự-lang Hàn-lâm-viện Hiệu-lý, « thần » Bùi-Sĩ-Tiêm
GT : Cẩn-sự-lang Hàn-lâm-viện Hiệu-lý, « thần » Bùi-Sĩ-Tiêm
PR : dệ vật gì, chùng loại gì số lượng bao nhiêu, do nha giữ đổ vật ấy
GT : đệ vật gì, chủng loại gì số lượng bao nhiêu, do nha giữ đồ vật ấy
PR : sức thực hành, cốt mong cho công việc được nhanh chóng xong xuôi
GT : sức thực hành, cốt mong cho công việc được nhanh chóng xong xuôi
PR : gần hơn thì quản giải được 2 tháng tiên lương, suất đội trưởng
GT : gần hơn thì quản giải được 2 tháng tiền lương, suất đội trưởng
PR : đời sống nhân dân và tăng cường chi viện cho miền Nam.
GT : đời sống nhân dân và tăng cường chi viện cho miền Nam.


## 6. Training loop

Replicates the real inference computation graph from `surya.recognition.batch_recognition`
(NOT `OCREncoderDecoderModel.forward`, which skips the text encoder step):

1. `encoder_hidden_states = model.encoder(pixel_values).last_hidden_state`
2. `encoder_text_hidden_states = model.text_encoder(query_token_ids, encoder_hidden_states=...).hidden_states`
3. `decoder_input_ids = shift_right(labels)` (teacher forcing)
4. `logits = model.decoder(decoder_input_ids, encoder_hidden_states=encoder_text_hidden_states, use_cache=False, prefill=True)`
5. cross-entropy loss against `labels`, ignoring `pad_id`

In [11]:
import torch.nn.functional as F
from tqdm.auto import tqdm


def shift_tokens_right(input_ids: torch.Tensor, pad_token_id: int, decoder_start_token_id: int) -> torch.Tensor:
    shifted = input_ids.new_zeros(input_ids.shape)
    shifted[:, 1:] = input_ids[:, :-1].clone()
    shifted[:, 0] = decoder_start_token_id
    shifted.masked_fill_(shifted == -100, pad_token_id)
    return shifted


def compute_logits(model, pixel_values, decoder_input_ids, query_token_count, device, dtype):
    pixel_values = pixel_values.to(device, dtype=dtype)
    decoder_input_ids = decoder_input_ids.to(device)
    batch_size = pixel_values.shape[0]

    # Cross-attention layers pre-allocate KV buffers sized for this batch, even with use_cache=False
    model.decoder.model._setup_cache(model.config, batch_size, device, dtype)
    model.text_encoder.model._setup_cache(model.config, batch_size, device, dtype)

    encoder_hidden_states = model.encoder(pixel_values=pixel_values).last_hidden_state

    batch_size = pixel_values.shape[0]
    text_encoder_input_ids = (
        torch.arange(query_token_count, device=device)
        .unsqueeze(0)
        .expand(batch_size, -1)
    )
    encoder_text_hidden_states = model.text_encoder(
        input_ids=text_encoder_input_ids,
        cache_position=None,
        attention_mask=None,
        encoder_hidden_states=encoder_hidden_states,
        encoder_attention_mask=None,
        use_cache=False,
    ).hidden_states

    seq_len = decoder_input_ids.shape[1]
    cache_position = torch.arange(seq_len, device=device)
    decoder_out = model.decoder(
        input_ids=decoder_input_ids,
        cache_position=cache_position,
        encoder_hidden_states=encoder_text_hidden_states,
        use_cache=False,
        prefill=True,
    )
    return decoder_out.logits


optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad), lr=LEARNING_RATE
)

global_step = 0
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{NUM_EPOCHS}")
    for pixel_values, labels in progress:
        labels = labels.to(DEVICE)
        decoder_input_ids = shift_tokens_right(labels, pad_id, decoder_start_token_id)

        logits = compute_logits(model, pixel_values, decoder_input_ids, query_token_count, DEVICE, DTYPE)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.shape[-1]).float(),
            labels.reshape(-1),
            ignore_index=pad_id,
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        global_step += 1
        if global_step % LOG_EVERY == 0:
            progress.set_postfix(loss=running_loss / LOG_EVERY)
            running_loss = 0.0


    epoch_dir = CHECKPOINT_DIR / f"epoch_{epoch + 1}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(epoch_dir)
    processor.save_pretrained(epoch_dir)
    print(f"Saved checkpoint to {epoch_dir}")

epoch 1/3:   0%|          | 0/11749 [00:00<?, ?it/s]

Saved checkpoint to /workspace/SinoNom-NLP/output/suryaocr_finetune/checkpoints/epoch_1


epoch 2/3:   0%|          | 0/11749 [00:00<?, ?it/s]

Saved checkpoint to /workspace/SinoNom-NLP/output/suryaocr_finetune/checkpoints/epoch_2


epoch 3/3:   0%|          | 0/11749 [00:00<?, ?it/s]

Saved checkpoint to /workspace/SinoNom-NLP/output/suryaocr_finetune/checkpoints/epoch_3


## 7. Validation - character error rate (CER)

Uses full autoregressive decoding (via `surya.recognition.batch_recognition`) on a subset
of the validation set, since teacher-forced loss alone does not reflect real inference quality.

In [14]:
import random
from rapidfuzz.distance import Levenshtein
from surya.recognition import batch_recognition

model.eval()

val_subset = val_examples[:VAL_SUBSET_SIZE] if VAL_SUBSET_SIZE else val_examples
val_images = [Image.open(p).convert("RGB") for p, _ in val_subset]
val_texts = [t for _, t in val_subset]
val_langs = [[LANG]] * len(val_subset)

with torch.no_grad():
    predictions, confidences = batch_recognition(val_images, val_langs, model, processor)

total_chars = 0
total_dist = 0
total_correct = 0
for pred_text, gt in zip(predictions, val_texts):
    edit_dist = Levenshtein.distance(pred_text, gt)
    total_dist += edit_dist
    total_chars += max(len(gt), 1)
    total_correct += int(pred_text == gt)

cer = total_dist / total_chars
acc = total_correct / len(predictions)
norm_edit = total_dist / sum(max(len(gt), 1) for gt in val_texts)
print(f"Validation CER on {len(val_subset)} samples: {cer:.4f}")

print(f"Validation Accuracy on {len(val_subset)} samples: {acc:.4f}")    
print("---")

print(f"Validation Normalized Edit Distance on {len(val_subset)} samples: {norm_edit:.4f}")    
print("PR :", pred_text)

for pred_text, gt in list(zip(predictions, val_texts))[:5]:
    print("PR :", pred_text)
    print("GT :", gt)
    print("---")


Recognizing Text:   0%|          | 0/1 [00:00<?, ?it/s]

Recognizing Text: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]

Validation CER on 200 samples: 0.0428
Validation Accuracy on 200 samples: 0.4700
---
Validation Normalized Edit Distance on 200 samples: 0.0428
PR : HỘI ĐIỂN SỰ LÊNAHA
PR : tiếng nối nghiệp nhà Hán là các bậc túc nho, đức lớn chen
GT : tiếng nối nghiệp nhà Hán là các bậc túc nho, đức lớn chen
---
PR : muộn khôn xiết. nhưng phụng thừa mệnh Thiên tử, không dám không
GT : muộn khôn xiết. nhưng phụng thừa mệnh Thiên tử, không dám không
---
PR : Dịch van :
GT : Dịch van:
---
PR : tháng. Còn các tên lái thuyền đều thưởng mỗi người 3 quan tiền.
GT : tháng. Còn các tên lái thuyền đều thưởng mỗi người 3 quan tiền.
---
PR : Khi vận nước hưng thịnh thì Hoắc Khú Bệnh(2) còn
GT : Khi vận nước hưng thịnh thì Hoắc Khứ Bệnh(2) còn
---


## Notes / limitations

- This fine-tunes `surya-ocr==0.8.0`'s recognition model only (no detection/layout training).
- The training loop is unofficial: it was reverse-engineered from the library's own inference
  code (`surya/recognition.py`) because Datalab never published a training script, even for
  this older version.
- To resume, point `load_rec_model()` at a saved `CHECKPOINT_DIR / "epoch_N"` directory instead
  of the default pretrained checkpoint.
- Consider unfreezing the encoder (`FREEZE_ENCODER = False`) with a lower learning rate for a
  full fine-tune once the decoder/text-encoder-only run converges.
- The current `surya-ocr` v2 (VLM-based) package is a completely different architecture and is
  not compatible with checkpoints produced by this notebook.